In [3]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset import IMDBDataset, get_celeb_name
from model import AgeClassifier
from torch.utils.data import random_split, SubsetRandomSampler
from torch import optim, nn
from tqdm import tqdm
from unlearn import *
from utils import *
from sklearn.model_selection import train_test_split
import random

In [ ]:
teacher_checkpoint_path = 'IMDB_CROP_Pretrained_50000_Samples_Teacher.pt'
forget_checkpoint_path = 'IMDB_CROP_Pretrained_50000_Samples_Forget.pt'

### Unzip Dataset

In [5]:
import tarfile
import os

if not os.path.isdir('./imdb_crop'):
    with tarfile.open('./imdb_crop.tar', 'r') as tar:
        tar.extractall('./')

In [ ]:
dataset = IMDBDataset('./imdb.csv', './imdb_crop', 50000) # remove this for full dataset

train_idx, valid_idx = train_test_split(list(range(len(dataset))), test_size=0.2, random_state=42)

train_ds = Subset(dataset, train_idx)
valid_ds = Subset(dataset, valid_idx)

Balanced dataset: 10000 samples across 5 age classes
Unique celebrities: 709
Class distribution: {0: 2000, 1: 2000, 2: 2000, 3: 2000, 4: 2000}


In [ ]:
celeb_ids = torch.tensor(dataset.celeb_ids)
ages = torch.tensor(dataset.ages)

unique_celebs = torch.unique(celeb_ids)
valid_celebs = []

for celeb_id in unique_celebs:
    if torch.any(ages[celeb_ids == celeb_id] <= 13):
        valid_celebs.append(celeb_id.item())

num_celeb = 5
if len(valid_celebs) < num_celeb:
    raise ValueError(f"Only {len(valid_celebs)} celebrities have images where age <= 13, but {num_celeb} requested")

forget_celeb_ids = random.sample(valid_celebs[:10], num_celeb)

In [12]:
print('forget celeb names:')
print(*[get_celeb_name(celeb_id) for celeb_id in forget_celeb_ids], sep='\n')

forget celeb names:
['A.J. Trauth']
['Abdallah El Akal']
['Adair Tishler']


In [13]:
retain_train_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i].item() not in forget_celeb_ids]))
retain_valid_ds = Subset(dataset, torch.tensor([i for i in valid_idx if celeb_ids[i].item() not in forget_celeb_ids]))

forget_train_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i].item() in forget_celeb_ids]))
forget_valid_ds = Subset(dataset, torch.tensor([i for i in valid_idx if celeb_ids[i].item() in forget_celeb_ids]))

In [14]:
print('dataset sizes:')
print(len(retain_train_ds), len(retain_valid_ds), len(forget_train_ds), len(forget_valid_ds), sep='\n')

dataset sizes:
7978
1991
22
9


In [15]:
device = 'cuda'

batch_size = 256
num_workers = 4

train_dl = DataLoader(train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

retain_train_dl = DataLoader(retain_train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
retain_valid_dl = DataLoader(retain_valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

forget_train_dl = DataLoader(forget_train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
forget_valid_dl = DataLoader(forget_valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

In [ ]:

full_trained_teacher = AgeClassifier(num_classes = 5, pretrained = True).to(device)

# Training
history = fit_one_cycle(10, full_trained_teacher, train_dl, valid_dl, device = device)

# Loading
# full_trained_teacher.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt", map_location = device))

# Saving
torch.save(full_trained_teacher.state_dict(), teacher_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarn

Epoch [0], last_lr: 0.01000, train_loss: 2.0897, val_loss: 2.1095, val_acc: 20.5454
Epoch [1], last_lr: 0.01000, train_loss: 1.7067, val_loss: 2.2578, val_acc: 21.3154
Epoch [2], last_lr: 0.01000, train_loss: 1.6977, val_loss: 1.6382, val_acc: 18.9716
Epoch [3], last_lr: 0.01000, train_loss: 1.7179, val_loss: 6.4824, val_acc: 18.7800
Epoch [4], last_lr: 0.01000, train_loss: 1.6890, val_loss: 2.8897, val_acc: 18.8251


In [17]:
evaluate(full_trained_teacher, retain_valid_dl, device)

{'Loss': 2.896615505218506,
 'Acc': 18.618728637695312,
 'Distribution': {'predictions': [85.08287048339844,
   0.0,
   14.866900444030762,
   0.05022601783275604,
   0.0],
  'labels': [18.88498306274414,
   18.08136558532715,
   21.496734619140625,
   20.09040641784668,
   21.446510314941406],
  'pred_counts': [1694, 0, 296, 1, 0],
  'label_counts': [376, 360, 428, 400, 427]}}

In [ ]:
evaluate(full_trained_teacher, forget_valid_dl, device)

{'Loss': 1.2326769828796387,
 'Acc': 44.40586471557617,
 'Distribution': {'predictions': [0.0,
   0.0,
   2.0281689167022705,
   97.97183227539062,
   0.0],
  'labels': [0.0,
   5.915493011474609,
   25.915491104125977,
   44.61971664428711,
   23.54929542541504],
  'pred_counts': [0, 0, 36, 1739, 0],
  'label_counts': [0, 105, 460, 792, 418]}}

### Forget

In [ ]:
model = AgeClassifier(num_classes = 5, pretrained = False).to(device)
unlearning_teacher = AgeClassifier(num_classes = 5, pretrained = False).to(device)

# Training
model.load_state_dict(torch.load(teacher_checkpoint_path, map_location = device))
blindspot_unlearner(model = model, unlearning_teacher = unlearning_teacher, full_trained_teacher = full_trained_teacher, 
                    retain_data = retain_train_ds, forget_data = forget_train_ds, epochs = 2, lr = 0.0001, 
                    batch_size = batch_size, num_workers = num_workers, device = device)

# Loading
# model.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_Forget_Class69_1_Epochs.pt", map_location = device))

# Saving
torch.save(model.state_dict(), forget_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1 Unlearning Loss 0.9879820346832275
Epoch 2 Unlearning Loss 0.7714890241622925
Epoch 3 Unlearning Loss 0.6557416915893555
Epoch 4 Unlearning Loss 0.5309666395187378
Epoch 5 Unlearning Loss 0.4359736442565918


In [ ]:
evaluate(model, retain_valid_dl, device)

{'Loss': 1.3399330377578735,
 'Acc': 38.222225189208984,
 'Distribution': {'predictions': [0.0, 0.0, 0.0, 100.0],
  'labels': [2.222222328186035,
   16.44444465637207,
   43.11111068725586,
   38.222225189208984],
  'pred_counts': [0, 0, 0, 225],
  'label_counts': [5, 37, 97, 86]}}

In [ ]:
evaluate(model, forget_valid_dl, device)

{'Loss': 1.3717129230499268,
 'Acc': 44.57327651977539,
 'Distribution': {'predictions': [0.0,
   0.11267606168985367,
   0.056338030844926834,
   99.83098602294922,
   0.0],
  'labels': [0.0,
   5.915493011474609,
   25.915491104125977,
   44.61971664428711,
   23.54929542541504],
  'pred_counts': [0, 2, 1, 1772, 0],
  'label_counts': [0, 105, 460, 792, 418]}}